# INR Network Parameter Comparison
Compare parameter counts across EfficientKAN, MLP, and SIREN implicit neural representation networks.

In [1]:
import torch
from networks import INR_Base

def count_parameters(model):
    """Count total trainable parameters in a model."""
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

# Network configuration (same for all networks for fair comparison)
config = {
    'n_input_dims': 3,
    'n_output_dims': 1,
    'n_hidden_layers': 3,
    'n_neurons': 64,
    'log2_hashmap_size': 0,  # Disable hash encoder to compare just the networks
}

# Create network instances
efficientkan_inr = INR_Base(network_type='efficientkan', **config)
mlp_inr = INR_Base(network_type='mlp', **config)
siren_inr = INR_Base(network_type='siren', **config)

# Calculate parameter counts
kan_params = count_parameters(efficientkan_inr)
mlp_params = count_parameters(mlp_inr)
siren_params = count_parameters(siren_inr)

# Print comparison
print("=" * 50)
print("INR Network Parameter Comparison")
print("=" * 50)
print(f"Configuration: {config['n_hidden_layers']} hidden layers, {config['n_neurons']} neurons each")
print(f"Input dims: {config['n_input_dims']}, Output dims: {config['n_output_dims']}")
print("-" * 50)
print(f"{'Network':<20} {'Parameters':>15} {'Ratio vs MLP':>15}")
print("-" * 50)
print(f"{'MLP':<20} {mlp_params:>15,} {1.0:>15.2f}x")
print(f"{'SIREN':<20} {siren_params:>15,} {siren_params/mlp_params:>15.2f}x")
print(f"{'EfficientKAN':<20} {kan_params:>15,} {kan_params/mlp_params:>15.2f}x")
print("=" * 50)

INR Network Parameter Comparison
Configuration: 3 hidden layers, 64 neurons each
Input dims: 3, Output dims: 1
--------------------------------------------------
Network                   Parameters    Ratio vs MLP
--------------------------------------------------
MLP                            8,448            1.00x
SIREN                          8,641            1.02x
EfficientKAN                 109,824           13.00x


In [3]:
# Find KAN configuration that matches MLP parameter count
# 
# MLP params per layer: out_features * in_features (no bias)
# KAN params per layer: out_features * in_features * (2 + grid_size + spline_order)
#   - base_weight: out * in
#   - spline_weight: out * in * (grid_size + spline_order)  
#   - spline_scaler: out * in
#
# Ratio: KAN/MLP ≈ (2 + grid_size + spline_order) per connection
# Default: grid_size=8, spline_order=3 → 13x more params

def calc_mlp_params(n_input, n_output, n_hidden_layers, n_neurons):
    """Calculate MLP parameter count (no bias)."""
    params = n_input * n_neurons  # first layer
    params += (n_hidden_layers - 1) * n_neurons * n_neurons  # hidden layers
    params += n_neurons * n_output  # output layer
    return params

def calc_kan_params(n_input, n_output, n_hidden_layers, n_neurons, grid_size=8, spline_order=3):
    """Calculate EfficientKAN parameter count."""
    multiplier = 2 + grid_size + spline_order  # params per connection vs MLP
    return calc_mlp_params(n_input, n_output, n_hidden_layers, n_neurons) * multiplier

# Target: match MLP with 3 hidden layers, 64 neurons
target_params = calc_mlp_params(3, 1, 3, 64)
print(f"Target MLP params: {target_params:,}")
print()

# Option 1: Reduce n_neurons for KAN (keep default grid_size=8, spline_order=3)
print("Option 1: Reduce neurons (grid_size=8, spline_order=3)")
print("-" * 50)
for n_neurons in [8, 12, 16, 18, 20, 24]:
    kan_p = calc_kan_params(3, 1, 3, n_neurons, grid_size=8, spline_order=3)
    print(f"  n_neurons={n_neurons:2d}: {kan_p:>8,} params (ratio: {kan_p/target_params:.2f}x)")

print()

# Option 2: Reduce grid_size (keep 64 neurons)
print("Option 2: Reduce grid_size (n_neurons=64, spline_order=3)")
print("-" * 50)
for grid_size in [1, 2, 3, 4, 5]:
    kan_p = calc_kan_params(3, 1, 3, 64, grid_size=grid_size, spline_order=3)
    multiplier = 2 + grid_size + 3
    print(f"  grid_size={grid_size}: {kan_p:>8,} params (ratio: {kan_p/target_params:.2f}x, multiplier={multiplier})")

print()

# Option 3: Combined - find exact match
print("Option 3: Find configurations close to target")
print("-" * 50)
best_configs = []
for n_neurons in range(8, 65):
    for grid_size in range(1, 16):
        kan_p = calc_kan_params(3, 1, 3, n_neurons, grid_size=grid_size, spline_order=3)
        ratio = kan_p / target_params
        if 0.95 <= ratio <= 1.05:
            best_configs.append((n_neurons, grid_size, kan_p, ratio))

best_configs.sort(key=lambda x: abs(x[3] - 1.0))
for n_neurons, grid_size, kan_p, ratio in best_configs[:5]:
    print(f"  n_neurons={n_neurons:2d}, grid_size={grid_size:2d}: {kan_p:>8,} params (ratio: {ratio:.3f}x)")

Target MLP params: 8,448

Option 1: Reduce neurons (grid_size=8, spline_order=3)
--------------------------------------------------
  n_neurons= 8:    2,080 params (ratio: 0.25x)
  n_neurons=12:    4,368 params (ratio: 0.52x)
  n_neurons=16:    7,488 params (ratio: 0.89x)
  n_neurons=18:    9,360 params (ratio: 1.11x)
  n_neurons=20:   11,440 params (ratio: 1.35x)
  n_neurons=24:   16,224 params (ratio: 1.92x)

Option 2: Reduce grid_size (n_neurons=64, spline_order=3)
--------------------------------------------------
  grid_size=1:   50,688 params (ratio: 6.00x, multiplier=6)
  grid_size=2:   59,136 params (ratio: 7.00x, multiplier=7)
  grid_size=3:   67,584 params (ratio: 8.00x, multiplier=8)
  grid_size=4:   76,032 params (ratio: 9.00x, multiplier=9)
  grid_size=5:   84,480 params (ratio: 10.00x, multiplier=10)

Option 3: Find configurations close to target
--------------------------------------------------
  n_neurons=22, grid_size= 3:    8,448 params (ratio: 1.000x)
  n_neurons=17

In [4]:
# Create matched KAN configuration and verify
# Best match: n_neurons=22, grid_size=3 gives exactly 8,448 params

from efficient_kan import KAN

# Matched KAN config
matched_kan_config = {
    'n_input_dims': 3,
    'n_output_dims': 1,
    'n_hidden_layers': 3,
    'n_neurons': 22,  # Reduced from 64
    'log2_hashmap_size': 0,
}

# Create matched KAN directly (need to pass grid_size which INR_Base doesn't expose well)
layers_hidden = [3] + [22] * 3 + [1]
matched_kan = KAN(layers_hidden=layers_hidden, grid_size=3, spline_order=3)

# Verify counts
matched_kan_params = count_parameters(matched_kan)

print("=" * 60)
print("Verified Parameter-Matched Network Comparison")
print("=" * 60)
print(f"{'Network':<25} {'Config':<25} {'Params':>10}")
print("-" * 60)
print(f"{'MLP':<25} {'64 neurons, 3 layers':<25} {mlp_params:>10,}")
print(f"{'SIREN':<25} {'64 neurons, 3 layers':<25} {siren_params:>10,}")
print(f"{'EfficientKAN (original)':<25} {'64 neurons, grid=8':<25} {kan_params:>10,}")
print(f"{'EfficientKAN (matched)':<25} {'22 neurons, grid=3':<25} {matched_kan_params:>10,}")
print("=" * 60)
print()
print("To use parameter-matched KAN in INR_Base, you'll need to modify")
print("EfficientKAN_Native to accept grid_size or use the KAN class directly.")

Verified Parameter-Matched Network Comparison
Network                   Config                        Params
------------------------------------------------------------
MLP                       64 neurons, 3 layers           8,448
SIREN                     64 neurons, 3 layers           8,641
EfficientKAN (original)   64 neurons, grid=8           109,824
EfficientKAN (matched)    22 neurons, grid=3             8,448

To use parameter-matched KAN in INR_Base, you'll need to modify
EfficientKAN_Native to accept grid_size or use the KAN class directly.


In [5]:
# Find EfficientKAN configs that maintain equal params across n_hidden_layers sweep [1, 5]
#
# For EfficientKAN with layers_hidden = [3] + [n]*L + [1]:
# - L+1 KANLinear layers
# - multiplier = 2 + grid_size + spline_order = 5 + grid_size
# 
# Total params = multiplier * [3*n + (L-1)*n² + n]
#             = multiplier * n * [4 + (L-1)*n]

def calc_efficient_kan_params(n_input, n_output, n_hidden_layers, n_neurons, grid_size, spline_order=3):
    """Calculate EfficientKAN parameter count."""
    multiplier = 2 + grid_size + spline_order
    L = n_hidden_layers
    n = n_neurons
    # First layer: n_input -> n_neurons
    # Hidden layers: (L-1) layers of n_neurons -> n_neurons
    # Output layer: n_neurons -> n_output
    params = multiplier * (n_input * n + (L - 1) * n * n + n * n_output)
    return params

def find_matching_configs(target_params, n_input=3, n_output=1, spline_order=3, 
                          max_neurons=128, max_grid_size=50):
    """Find (n_neurons, grid_size) pairs for each n_hidden_layers that match target."""
    results = {}
    for L in range(1, 6):
        best = None
        best_diff = float('inf')
        for n in range(4, max_neurons + 1):
            for gs in range(1, max_grid_size + 1):
                params = calc_efficient_kan_params(n_input, n_output, L, n, gs, spline_order)
                diff = abs(params - target_params)
                if diff < best_diff:
                    best_diff = diff
                    best = (n, gs, params)
                if params == target_params:
                    break
        results[L] = best
    return results

# Use L=3, n=16, grid_size=5 as reference (from your current config)
ref_params = calc_efficient_kan_params(3, 1, 3, 16, 5)
print(f"Reference params (L=3, n=16, grid_size=5): {ref_params:,}")
print()

# Find matching configs for all L values
matches = find_matching_configs(ref_params)

print("Parameter-matched EfficientKAN configs for n_hidden_layers sweep [1, 5]:")
print("=" * 70)
print(f"{'n_hidden_layers':<18} {'n_neurons':<12} {'grid_size':<12} {'params':<12} {'diff':<10}")
print("-" * 70)
for L in range(1, 6):
    n, gs, p = matches[L]
    diff = p - ref_params
    diff_str = f"{diff:+d}" if diff != 0 else "exact"
    print(f"{L:<18} {n:<12} {gs:<12} {p:<12,} {diff_str:<10}")
print("=" * 70)

Reference params (L=3, n=16, grid_size=5): 5,760

Parameter-matched EfficientKAN configs for n_hidden_layers sweep [1, 5]:
n_hidden_layers    n_neurons    grid_size    params       diff      
----------------------------------------------------------------------
1                  30           43           5,760        exact     
2                  12           25           5,760        exact     
3                  8            31           5,760        exact     
4                  12           7            5,760        exact     
5                  5            43           5,760        exact     


In [ ]:
# Generate the JSON structure for hidden_layer_sweep-param-matched.json
# Note: EfficientKAN uses 'grid_size', not 'num_grids' (that's FastKAN)

import json

datasets = [
    "beechnut", "chameleon", "magnetic_reconnection", "pawpawsaurus",
    "hcci_oh", "kingsnake", "marmoset_neurons", "spathorhynchus", 
    "statue_leg", "frog"
]

# Parameter-matched configs for EfficientKAN
# Each n_hidden_layers value gets its own (n_neurons, grid_size) to maintain 5,760 params
param_matched_configs = {
    1: {"n_neurons": 30, "grid_size": 43},
    2: {"n_neurons": 12, "grid_size": 25},
    3: {"n_neurons": 8,  "grid_size": 31},
    4: {"n_neurons": 12, "grid_size": 7},
    5: {"n_neurons": 5,  "grid_size": 43},
}

# Build the sweep JSON - now with separate entries for each n_hidden_layers
sweep_config = {}
for dataset in datasets:
    sweep_config[dataset] = []
    for n_hidden_layers, kan_cfg in param_matched_configs.items():
        sweep_config[dataset].append({
            "lrate": 0.004,
            "lrate_decay": 16,
            "epochs": 24,
            "n_neurons": kan_cfg["n_neurons"],
            "n_hidden_layers": n_hidden_layers,
            "log2_hashmap_size": 0,
            "kan_params": {
                "grid_size": kan_cfg["grid_size"]
            }
        })

print("Updated hidden_layer_sweep-param-matched.json for EfficientKAN:")
print(json.dumps(sweep_config, indent=4))